# MLL23 classifier baseline on Kaggle (free GPU)

End-to-end Stage A + M2 baseline: verify data, extract DinoBloom-B embeddings on GPU,
train all three heads, compare, calibrate.

**Before running:**

- Settings → Accelerator → **GPU** (T4). Confirm with the cell below.
- Settings → Internet → **ON** (needed once: clone + weights download).
- Add data → upload the 18 `*.zip` files from Zenodo record 14277609 as a private
  dataset (e.g. `mll23-zips`) and attach it. Nothing else is needed on Kaggle.

In [ ]:
import torch

assert torch.cuda.is_available(), "enable a GPU accelerator first"
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
!git clone --depth 1 https://github.com/3r3bu5/blood-analyzer.git /kaggle/working/blood
%cd /kaggle/working/blood
!pip install -q timm torchvision Pillow numpy PyYAML

## Stage the dataset from the attached Kaggle dataset

Unzip the 18 class archives, then let the project's own verifier confirm every
MD5 against the registry (offline — the checksums ship in the repo).

In [ ]:
import glob
import os
import subprocess

zips = sorted(glob.glob("/kaggle/input/mll23-zips/*.zip"))
assert len(zips) == 18, f"expected 18 zips, found {len(zips)}"
os.makedirs("data/raw/MLL23", exist_ok=True)
for archive in zips:
    print("extracting", os.path.basename(archive), flush=True)
    subprocess.run(["unzip", "-q", "-o", archive, "-d", "data/raw/MLL23"], check=True)

In [ ]:
!PYTHONPATH=src python -m bloodfilm.cli assets verify mll23 --report-output outputs/reports/mll23_verify.json
!PYTHONPATH=src python -m bloodfilm.cli dataset build-manifest mll23 --output data/manifests/mll23_manifest.csv --invalid-output data/manifests/mll23_invalid.csv --report-output outputs/reports/mll23_manifest.json
!PYTHONPATH=src python -m bloodfilm.cli dataset split --manifest data/manifests/mll23_manifest.csv --output data/manifests/mll23_splits.csv --report-output outputs/reports/mll23_leakage.json --seed 42

## DinoBloom-B weights (Apache-2.0, HuggingFace MarrLab/DinoBloom)

In [ ]:
!mkdir -p models/backbones
!curl -L -o models/backbones/dinobloom-b.pth "https://huggingface.co/MarrLab/DinoBloom/resolve/main/pytorch_model_b.bin"
!echo "fdd8e51fd9831858c96cb241a694dd9d64cc28941ec2a923f1fa1e9b5c610a3f  models/backbones/dinobloom-b.pth" | sha256sum -c

In [ ]:
import json
from pathlib import Path

WEIGHT_SHA256 = "fdd8e51fd9831858c96cb241a694dd9d64cc28941ec2a923f1fa1e9b5c610a3f"
path = Path("configs/registry/assets.yaml")
registry = json.loads(path.read_text(encoding="utf-8"))
model = registry["models"][0]
assert model["name"] == "DinoBloom-B"
model["sha256"] = WEIGHT_SHA256
model["license"] = "Apache-2.0"
model["status"] = "downloaded_pending_smoke"
path.write_text(json.dumps(registry, indent=2), encoding="utf-8")
print("registry updated")

In [ ]:
!PYTHONPATH=src python -m bloodfilm.cli smoke --config configs/base.yaml --image data/raw/MLL23/basophil/basophil/basophil_0001.TIF

## Stage A: extract embeddings (`--device auto` picks the GPU)

Expect roughly half an hour for all 41,621 crops on a T4.

In [ ]:
!PYTHONPATH=src python -m bloodfilm.cli embeddings extract --config configs/classifier_mll23.yaml --device auto --output data/embeddings/mll23_embeddings.pt --report-output outputs/reports/mll23_embeddings.json

## M2: train all heads, compare, calibrate

In [ ]:
!PYTHONPATH=src python -m bloodfilm.cli train classifier --config configs/classifier_mll23.yaml --head all --epochs 50 --checkpoint-dir outputs/checkpoints --report-output outputs/reports/mll23_head_comparison.json
import json
from pathlib import Path

comparison = json.loads(Path("outputs/reports/mll23_head_comparison.json").read_text(encoding="utf-8"))
for head, metrics in comparison["heads"].items():
    print(f"{head:8s} macro_f1={metrics['macro_f1']:.3f} acc={metrics['accuracy']:.3f} ece={metrics['ece_before']:.3f}->{metrics['ece_after']:.3f}")
print("selected:", comparison["selected"])

## Takeaways

Download from this session before it expires (File → Download all, or save the
small files): `data/embeddings/mll23_embeddings.pt` (~128 MB), `outputs/checkpoints/`
and `outputs/reports/`. The zips stay in your Kaggle dataset; the weights re-download
in one command — so a fresh session reproduces everything above.